In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('homework') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/06 10:52:00 WARN Utils: Your hostname, Spencers-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.86 instead (on interface en0)
26/02/06 10:52:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/06 10:52:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
spark.version

'4.1.1'

!wget -O https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet > 'data/raw/yellow/2024/10.parquet'

Doesn't work:
df = spark.read.parquet("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet")

In [7]:
local_path = "data/raw/yellow/2024/10/yellow_trip_data_2024_10.parquet"

In [8]:
import requests
import os

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet"
# local_path = "/data/raw/yellow/2024/10/yellow_trip_data_2024_10.parquet"

# Create directory structure if it doesn't exist
os.makedirs(os.path.dirname(local_path), exist_ok=True)

with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open(local_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

print("Download complete")


Download complete


In [10]:
from pyspark.sql import types

In [11]:
yellow_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.IntegerType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.IntegerType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("payment_type", types.IntegerType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

In [14]:
# input_path = f'data/raw/yellow/{year}/{month:02d}/'
output_path = 'data/pq/yellow/2024/10/'

# .option("header", "true") \
# .schema(yellow_schema) \
df_yellow = spark.read.parquet(local_path)

df_yellow \
    .repartition(4) \
    .write.parquet(output_path)

See job in Spark dashboard. "Output size" is about 22.4MiB / ~1M records (partition)

In [17]:
df_yellow = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

In [23]:
df_yellow.registerTempTable('yellow')

/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/python/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [24]:
q = "SELECT COUNT(*) FROM yellow WHERE pickup_datetime >= '2024-10-15' AND pickup_datetime < '2024-10-16';"

In [25]:
spark.sql(q).show()

+--------+
|count(1)|
+--------+
|  128893|
+--------+



In [44]:
longest_trip_q = "\
    SELECT \
        MAX( \
            EXTRACT(HOURS FROM dropoff_datetime-pickup_datetime) \
            + (EXTRACT(DAYS FROM dropoff_datetime-pickup_datetime) * 24) \
        ) AS max_dur \
    FROM yellow;
"

In [45]:
spark.sql(longest_trip_q).show()

+-------+
|max_dur|
+-------+
|    162|
+-------+



download the zone data to join zone name on zone ID

In [46]:
zone_path = "data/raw/taxi_zone_lookup.csv"

In [49]:
!wget -O data/raw/taxi_zone_lookup.csv https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv 

--2026-02-06 12:20:34--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.35.25.78, 13.35.25.141, 13.35.25.186, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.35.25.78|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘data/raw/taxi_zone_lookup.csv’

data/raw/taxi_zone_ 100%[===================>]  12.04K  --.-KB/s    in 0.01s   

2026-02-06 12:20:34 (1.07 MB/s) - ‘data/raw/taxi_zone_lookup.csv’ saved [12331/12331]



In [51]:
df_test = spark.read.option("header", "true").csv(zone_path)
df_test.schema

StructType([StructField('LocationID', StringType(), True), StructField('Borough', StringType(), True), StructField('Zone', StringType(), True), StructField('service_zone', StringType(), True)])

In [52]:
from pyspark.sql import types

In [53]:
zone_schema = types.StructType([
    types.StructField("LocationID", types.StringType(), True),
    types.StructField("Borough", types.StringType(), True),
    types.StructField("Zone", types.StringType(), True),
    types.StructField("service_zone", types.StringType(), True),
])

In [54]:
df_zone = spark.read \
    .option("header", "true") \
    .schema(zone_schema) \
    .csv(zone_path)

In [63]:
df_zone.show(10)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 10 rows


Using the zone lookup data and the Yellow October 2024 data, what is the name of the LEAST frequent pickup location Zone?

- join zone data on LocationID
- query least frequent pickup zone

In [55]:
df_zone.registerTempTable('zone')

In [64]:
df_least_pickup = spark.sql("""
SELECT
    PULocationID AS LocationId,
    COUNT(1) AS number_records
FROM yellow
GROUP BY 1
ORDER BY number_records ASC
LIMIT 10;
""")

In [65]:
df_join = df_least_pickup.join(df_zone, on=['LocationID'], how='inner')

In [66]:
df_join.show()

+----------+--------------+-------------+--------------------+------------+
|LocationId|number_records|      Borough|                Zone|service_zone|
+----------+--------------+-------------+--------------------+------------+
|         2|             3|       Queens|         Jamaica Bay|   Boro Zone|
|         5|             2|Staten Island|       Arden Heights|   Boro Zone|
|        44|             4|Staten Island|Charleston/Totten...|   Boro Zone|
|        84|             4|Staten Island|Eltingville/Annad...|   Boro Zone|
|       105|             1|    Manhattan|Governor's Island...| Yellow Zone|
|       111|             3|     Brooklyn| Green-Wood Cemetery|   Boro Zone|
|       187|             4|Staten Island|       Port Richmond|   Boro Zone|
|       199|             2|        Bronx|       Rikers Island|   Boro Zone|
|       204|             4|Staten Island|   Rossville/Woodrow|   Boro Zone|
|       245|             4|Staten Island|       West Brighton|   Boro Zone|
+----------+